# Phase 2T - The leakage effect as a function of similarity

**Run on:** Kaggle or Colab, free T4. ~55 min for 45 training runs.

---

### The gap this closes

Two measurements currently sit in the paper without a bridge between them:

| measurement | intervention | effect |
|---|---|---|
| Phase 2N | remove training images at cosine >= 0.98 | **7.3-7.6 points** |
| Phase 2X | partition by connected components at 0.90 | **24-37 points** |

They are not in conflict; they are different interventions. Grouping at 0.90
separates a far broader class of related images than "near-twin at 0.98", and it
does so transitively, through chaining. But a reader is owed an account of how
one becomes the other.

This notebook sweeps the deduplication threshold and measures the
size-controlled effect at each. All arms share **byte-identical test folds**, so
every number on the curve is directly comparable to every other and to the
image-level baseline.

### What the shape will tell us

- **Effect rises steadily as the threshold falls:** the grouped protocol's larger
  gap is the same mechanism measured at a looser notion of similarity. The
  decomposition is complete and the paper can state it as a curve.
- **Effect stays flat near 7 points and the grouped gap remains unexplained:**
  something other than pairwise similarity drives most of the grouped result,
  and the paper must say so rather than implying duplication accounts for it.

Either way this is the last measurement needed to describe the effect honestly.

## 1. Environment, data, similarity

In [ ]:
import subprocess, sys
for pkg in ["opencv-python-headless", "tabulate", "kagglehub"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg],
                   check=False)

import os, json, time, shutil, random, gc
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             balanced_accuracy_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("GPUs:", tf.config.list_physical_devices("GPU"))

IN_KAGGLE = os.path.exists("/kaggle/working")
WORK    = "/kaggle/working" if IN_KAGGLE else "/content"
SCRATCH = "/kaggle/temp"    if IN_KAGGLE else "/content"
try:
    os.makedirs(SCRATCH, exist_ok=True)
except OSError:
    SCRATCH = "/tmp"; os.makedirs(SCRATCH, exist_ok=True)

RESULTS_DIR = f"{WORK}/fyp_phase2t_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
N_FOLDS = 5
THRESHOLDS = [0.90, 0.94, 0.96, 0.98]
RESULTS = {"seed": SEED, "n_folds": N_FOLDS, "thresholds": THRESHOLDS}

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)
print("outputs ->", RESULTS_DIR)

In [ ]:
FOLDERS = {"Benign": "Bengin cases", "Malignant": "Malignant cases",
           "Normal": "Normal cases"}
DATA_ROOT = None
if os.path.isdir("/kaggle/input"):
    hits = [d for d, _, _ in os.walk("/kaggle/input")
            if os.path.basename(d) == "Malignant cases"]
    if hits:
        DATA_ROOT = os.path.dirname(hits[0])
if DATA_ROOT is None:
    import kagglehub
    DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
    cands = [d for d, _, _ in os.walk(DL) if os.path.basename(d) == "Malignant cases"]
    DATA_ROOT = os.path.dirname(cands[0])
print("data:", DATA_ROOT)

SPLIT_URL = ("https://raw.githubusercontent.com/haseebkhan9081/"
             "iqothnccd-leakage-audit/main/split_seed42.csv")
subprocess.run(["wget", "-q", "-O", f"{SCRATCH}/split_seed42.csv", SPLIT_URL],
               check=True)
df = pd.read_csv(f"{SCRATCH}/split_seed42.csv")
df["path"] = [os.path.join(DATA_ROOT, FOLDERS[l], f)
              for l, f in zip(df["label"], df["file"])]
assert all(os.path.exists(p) for p in df["path"])
y_all = df["y"].to_numpy()

thumbs = []
for p in tqdm(df["path"], desc="thumbnails"):
    g = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    g -= g.mean()
    n = np.linalg.norm(g)
    thumbs.append(g / n if n > 0 else g)
T = np.stack(thumbs)
S = T @ T.T
np.fill_diagonal(S, 0.0)
RESULTS["pairs_by_threshold"] = {
    str(t): int((np.triu(S, 1) >= t).sum()) for t in THRESHOLDS}
print("near-duplicate pairs in the dataset:", RESULTS["pairs_by_threshold"])
save_json()

## 2. Nine conditions, one set of test folds

The baseline plus, for each threshold, a dedup arm and a size-matched random
arm. Every condition is evaluated on the same five test folds.

In [ ]:
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                             random_state=SEED).split(df, y_all))
rng = np.random.default_rng(SEED)

CONDITIONS = {"image-level": [(tr, te) for tr, te in folds]}
removal_stats = []
for t in THRESHOLDS:
    dedup_folds, rand_folds = [], []
    for k, (tr, te) in enumerate(folds):
        contaminated = (S[np.ix_(tr, te)] >= t).any(axis=1)
        n_removed = int(contaminated.sum())
        dedup_folds.append((tr[~contaminated], te))
        rand_folds.append((np.sort(rng.choice(tr, size=len(tr) - n_removed,
                                              replace=False)), te))
        removal_stats.append({"threshold": t, "fold": k,
                              "n_train": int(len(tr)), "n_removed": n_removed,
                              "pct_removed": round(100 * n_removed / len(tr), 2)})
    CONDITIONS[f"dedup@{t}"] = dedup_folds
    CONDITIONS[f"random@{t}"] = rand_folds

RESULTS["removal_stats"] = removal_stats
for t in THRESHOLDS:
    pcts = [r["pct_removed"] for r in removal_stats if r["threshold"] == t]
    print(f"threshold {t}: {np.mean(pcts):5.1f}% of training images removed "
          f"(range {min(pcts):.1f}-{max(pcts):.1f}%)")

for name, cf in CONDITIONS.items():
    for k in range(N_FOLDS):
        assert np.array_equal(cf[k][1], folds[k][1]), f"{name} fold {k} test set differs"
for t in THRESHOLDS:
    for k in range(N_FOLDS):
        assert len(CONDITIONS[f"dedup@{t}"][k][0]) == len(CONDITIONS[f"random@{t}"][k][0])
print("\ntest folds identical everywhere: OK")
print("dedup and random training sizes matched at every threshold: OK")
save_json()

## 3. Run

In [ ]:
SIZE, EPOCHS, BATCH, LR = 224, 30, 16, 1e-4
X = np.empty((len(df), SIZE, SIZE, 3), np.float32)
for i, p in enumerate(tqdm(df["path"], desc=f"load {SIZE}px")):
    X[i] = np.asarray(Image.open(p).convert("RGB").resize((SIZE, SIZE),
                                                          Image.BILINEAR), np.float32)

def build():
    inp = layers.Input((SIZE, SIZE, 3))
    x = keras.Sequential([layers.RandomFlip("horizontal"),
                          layers.RandomRotation(0.05),
                          layers.RandomZoom(0.15, 0.15)], name="aug")(inp)
    base = keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=(SIZE, SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False
    x = base(keras.applications.efficientnet.preprocess_input(x), training=False)
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(inp, layers.Dense(3, activation="softmax")(x))

def evaluate(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1.mean())}

rows, t0all = [], time.time()
for cond, cf in CONDITIONS.items():
    for k, (tr, te) in enumerate(cf):
        keras.backend.clear_session()
        tf.random.set_seed(SEED + k)
        counts = np.bincount(y_all[tr], minlength=3)
        cw = {i: float(len(tr) / (3 * c)) if c else 0.0 for i, c in enumerate(counts)}
        model = build()
        model.compile(optimizer=keras.optimizers.Adam(LR),
                      loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        model.fit(X[tr], y_all[tr], epochs=EPOCHS, batch_size=BATCH,
                  class_weight=cw, verbose=0)
        m = evaluate(y_all[te], model.predict(X[te], verbose=0).argmax(1))
        m.update({"condition": cond, "fold": k, "n_train": int(len(tr))})
        rows.append(m)
        print(f"{cond:16s} fold {k} | bal {m['balanced_accuracy']:.3f}")
        del model; gc.collect()

cv = pd.DataFrame(rows)
cv.to_csv(f"{RESULTS_DIR}/cv_per_fold.csv", index=False)
RESULTS["per_fold"] = cv.to_dict("records")
print(f"\ntotal {(time.time()-t0all)/60:.1f} min over {len(cv)} runs")
save_json()

## 4. The curve

In [ ]:
def bal(cond):
    return cv[cv.condition == cond].sort_values("fold")["balanced_accuracy"].to_numpy()

base = bal("image-level")
curve = []
for t in THRESHOLDS:
    d, r = bal(f"dedup@{t}"), bal(f"random@{t}")
    eff = (base - d).mean() - (base - r).mean()
    sd = ((base - d) - (base - r)).std(ddof=1)
    pct = np.mean([s["pct_removed"] for s in removal_stats if s["threshold"] == t])
    curve.append({"threshold": t, "pct_train_removed": round(float(pct), 2),
                  "dedup_bal": float(d.mean()), "random_bal": float(r.mean()),
                  "size_controlled_effect": float(eff),
                  "effect_sd": float(sd)})
    print(f"threshold {t}: {pct:5.1f}% removed | dedup {d.mean():.4f} | "
          f"random {r.mean():.4f} | effect {eff:+.4f} (sd {sd:.4f})")

RESULTS["curve"] = curve
RESULTS["image_level_bal"] = float(base.mean())
save_json()
print()
print(f"image-level baseline: {base.mean():.4f}")
print("The size-controlled effect is the leakage attributable to similarity at")
print("that threshold. Compare its trend against the 24-37 point grouped gap.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ts = [c["threshold"] for c in curve]
axes[0].plot(ts, [c["pct_train_removed"] for c in curve], "o-", color="#7d3c98")
axes[0].set_xlabel("dedup threshold (cosine)"); axes[0].set_ylabel("% of training set removed")
axes[0].set_title("How much is removed"); axes[0].grid(alpha=.3); axes[0].invert_xaxis()

axes[1].errorbar(ts, [c["size_controlled_effect"] for c in curve],
                 yerr=[c["effect_sd"] for c in curve], fmt="o-", capsize=4,
                 color="#c0392b", label="size-controlled leakage effect")
axes[1].axhline(0, color="grey", lw=1)
axes[1].set_xlabel("dedup threshold (cosine)")
axes[1].set_ylabel("balanced-accuracy points")
axes[1].set_title("Leakage vs how strict 'duplicate' is")
axes[1].grid(alpha=.3); axes[1].legend(fontsize=8); axes[1].invert_xaxis()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_dedup_sweep.png", dpi=200)
plt.show()

In [ ]:
path = shutil.make_archive(f"{WORK}/fyp_phase2t_results", "zip", RESULTS_DIR)
print("archive:", path, f"({os.path.getsize(path)/1e6:.1f} MB)")
if not IN_KAGGLE:
    try:
        from google.colab import files; files.download(path)
    except Exception as e:
        print("download from the file browser:", e)